# Gate 32C - Basis-Convergence Audit for the Level 2 H_T Bound

This notebook audits finite-basis truncation stability for the Level 2 `H_T` proxy. It is convergence evidence for the implemented scaffold only; it is not the full analytic spectrum and does not prove the no-extra-light-state theorem.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from ht_operator import alpha_scaled_a, default_level2_config
from spectral_bounds import basis_convergence_scan
from spectral_gap import MU_H, natural_lambda2

k_values = [4, 6, 8, 10, 12, 16]
a_values = [0.573, 1.0, alpha_scaled_a()]
rows = basis_convergence_scan(k_values, a_values, default_level2_config, natural_lambda2())
len(rows), MU_H, natural_lambda2()

## Convergence Table

Each row records the direct finite-basis gap and lower-bound margins. The `passes` column requires the worst of the direct, Gershgorin, and min-max margins to clear the target.

In [ ]:
convergence_table = [
    {
        "k_max": row["k_max"],
        "a": row["a"],
        "basis_size": row["basis_size"],
        "zero_mode_count": row["zero_mode_count"],
        "first_d": row["first_complement_eigenvalue"],
        "ht_gap": row["ht_gap"],
        "direct_margin": row["direct_margin"],
        "gershgorin_margin": row["gershgorin_margin"],
        "minmax_margin": row["minmax_margin"],
        "worst_margin": row["worst_margin"],
        "passes": row["passes"],
        "monotonicity_notes": row["monotonicity_notes"],
    }
    for row in rows
]
convergence_table

## First Complement Eigenvalue vs k_max

The table below is a plot-equivalent audit table grouped by anisotropy.

In [ ]:
first_d_by_k = {}
for row in rows:
    first_d_by_k.setdefault(row["a"], []).append((row["k_max"], row["first_complement_eigenvalue"]))
first_d_by_k

## Lower-Bound Margins vs k_max

This table records the margin sequence for each lower-bound mode.

In [ ]:
margins_by_k = {}
for row in rows:
    margins_by_k.setdefault(row["a"], []).append(
        {
            "k_max": row["k_max"],
            "direct": row["direct_margin"],
            "gershgorin": row["gershgorin_margin"],
            "minmax": row["minmax_margin"],
            "worst": row["worst_margin"],
        }
    )
margins_by_k

## Worst-Case Row

In [ ]:
worst_row = min(rows, key=lambda row: row["worst_margin"])
worst_row

## Limitation

Gate 32C audits finite-basis convergence for the Level 2 proxy. Stable truncation behavior is useful evidence, but it is not the complete Hilbert-space spectral theorem. The full analytic `H_T` spectrum remains open.